In [1]:
import pandas as pd
from pathlib import Path
import unicodedata

# Carpeta origen
carpeta = Path(r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 2. PROYECTOS\HANDHELDS SPUE\Pruebas\Abril_Honeywell_Urovo+T1Pro\Surtido_hist")
ruta_pasillos = Path(r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 2. PROYECTOS\HANDHELDS SPUE\Pruebas\Abril_Honeywell_Urovo+T1Pro\Pasillo surtido.xlsx")

# Extensiones a leer
ext_validas = {".xlsx", ".xls", ".csv", ".txt"}

# Columnas objetivo para analisis
columnas_objetivo = {
    "fecha_hora": ["fecha hora", "fechahora", "fecha_hora", "fecha"],
    "usuario": ["usuario", "user"],
    "pedido": ["pedido", "orden", "order"],
    "codigo": ["codigo", "cdigo", "sku", "articulo", "producto"],
    "localidad": ["localidad", "ubicacion", "location"],
    "anden": ["anden", "and en", "dock", "puerta"]
}

registros = []
errores = []


def normalizar_texto(txt):
    txt = str(txt).strip().lower()
    txt = "".join(ch for ch in unicodedata.normalize("NFKD", txt) if not unicodedata.combining(ch))
    txt = " ".join(txt.replace("_", " ").split())
    return txt


def normalizar_clave(valor):
    if pd.isna(valor):
        return pd.NA
    texto = str(valor).strip()
    if texto.endswith(".0"):
        texto = texto[:-2]
    return texto


def mapear_columnas(cols):
    cols_norm = {c: normalizar_texto(c) for c in cols}
    mapeo = {}

    for canonica, aliases in columnas_objetivo.items():
        aliases_norm = {normalizar_texto(a) for a in aliases}
        for original, normalizada in cols_norm.items():
            if normalizada in aliases_norm:
                mapeo[canonica] = original
                break

    return mapeo


def leer_tabla(archivo, hoja=None):
    if archivo.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(archivo, sheet_name=hoja)

    if archivo.suffix.lower() == ".csv":
        try:
            return pd.read_csv(archivo, encoding="utf-8")
        except UnicodeDecodeError:
            return pd.read_csv(archivo, encoding="latin-1")

    # Ajusta sep si tu txt usa otro delimitador
    return pd.read_csv(archivo, sep=",", encoding="utf-8", engine="python")


for archivo in carpeta.iterdir():
    if not archivo.is_file() or archivo.suffix.lower() not in ext_validas:
        continue

    try:
        if archivo.suffix.lower() in [".xlsx", ".xls"]:
            xls = pd.ExcelFile(archivo)
            hojas = xls.sheet_names
        else:
            hojas = [None]

        for hoja in hojas:
            df = leer_tabla(archivo, hoja)
            mapeo = mapear_columnas(df.columns)

            detalle = pd.DataFrame({
                "archivo": archivo.name,
                "hoja": hoja if hoja is not None else "",
                "fecha_hora": df[mapeo["fecha_hora"]] if "fecha_hora" in mapeo else pd.NA,
                "usuario": df[mapeo["usuario"]] if "usuario" in mapeo else pd.NA,
                "pedido": df[mapeo["pedido"]] if "pedido" in mapeo else pd.NA,
                "codigo": df[mapeo["codigo"]] if "codigo" in mapeo else pd.NA,
                "localidad": df[mapeo["localidad"]] if "localidad" in mapeo else pd.NA,
                "anden": df[mapeo["anden"]] if "anden" in mapeo else pd.NA,
            })

            registros.append(detalle)

    except Exception as e:
        errores.append({"archivo": archivo.name, "hoja": hoja if 'hoja' in locals() else "", "error": str(e)})


# DF detalle con solo las columnas utiles
df_surtido = pd.concat(registros, ignore_index=True) if registros else pd.DataFrame(
    columns=["archivo", "hoja", "fecha_hora", "usuario", "pedido", "codigo", "localidad", "anden"]
)

# Limpieza minima
# dayfirst=True evita inversión día/mes al parsear fechas como texto (formato latino).
df_surtido["fecha_hora"] = pd.to_datetime(df_surtido["fecha_hora"], errors="coerce", dayfirst=True)
for col in ["usuario", "pedido", "codigo", "localidad", "anden"]:
    df_surtido[col] = df_surtido[col].astype("string").str.strip()

# Cruce Localidad -> Pasillo
df_surtido["localidad_clave"] = df_surtido["localidad"].map(normalizar_clave).astype("string")

df_pasillos = pd.read_excel(ruta_pasillos, sheet_name="PS", usecols=["Clave Localidad", "Pasillo"])
df_pasillos["Clave Localidad"] = df_pasillos["Clave Localidad"].map(normalizar_clave).astype("string")
df_pasillos["Pasillo"] = df_pasillos["Pasillo"].astype("string").str.strip()
df_pasillos = df_pasillos.dropna(subset=["Clave Localidad", "Pasillo"]).drop_duplicates(subset=["Clave Localidad"])

df_surtido = df_surtido.merge(
    df_pasillos.rename(columns={"Clave Localidad": "localidad_clave", "Pasillo": "pasillo"}),
    on="localidad_clave",
    how="left"
)

df_surtido = df_surtido.drop(columns=["localidad_clave"])

# DF resumen de las columnas clave
df_resumen_columnas = pd.DataFrame([
    {
        "columna": col,
        "no_nulos": int(df_surtido[col].notna().sum()),
        "nulos": int(df_surtido[col].isna().sum()),
        "valores_unicos": int(df_surtido[col].nunique(dropna=True))
    }
    for col in ["fecha_hora", "usuario", "pedido", "codigo", "localidad", "anden", "pasillo"]
])

print("DF detalle creado: df_surtido")
print(df_surtido.head())
print("\nDF resumen creado: df_resumen_columnas")
print(df_resumen_columnas)

if errores:
    df_errores = pd.DataFrame(errores)
    print("\nArchivos con error:")
    print(df_errores)

c:\Users\igcamposg\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


DF detalle creado: df_surtido
                  archivo     hoja          fecha_hora  \
0  Surtido abil01_20.xlsx  Surtido 2026-04-01 00:00:00   
1  Surtido abil01_20.xlsx  Surtido 2026-04-01 00:00:00   
2  Surtido abil01_20.xlsx  Surtido 2026-04-01 00:00:00   
3  Surtido abil01_20.xlsx  Surtido 2026-04-01 00:00:01   
4  Surtido abil01_20.xlsx  Surtido 2026-04-01 00:00:01   

                     usuario     pedido  \
0    980729 PUS0729 IRIGOYEN  762172490   
1     980884 PUS0884 DELGADO  762171030   
2  4105416 4105416 HERNANDEZ  762172486   
3    4030209 4030209 SERRANO  762171330   
4     4037077 4037077 TORRES  762172485   

                                              codigo localidad anden pasillo  
0  17629 HTA-105|HTA-105 Hilo redondo, 2.7mm x 12...    111726    12      12  
1  48022 CODO-MTAS|CODO-MTAS Contacto de tierra a...    103124     1       4  
2  48776 CG-236|CG-236 Reducción bushing de hierr...    110454    12      11  
3  16743 EST-35|EST-35 Escalera tipo tijera T3

In [2]:
# Analisis por dia y turno para identificar pasillos por usuario
# Requiere que df_surtido exista desde la celda 1.

import numpy as np

# Configuracion de turnos
HORA_INICIO_T1 = pd.to_timedelta("06:00:00")
HORA_FIN_T1 = pd.to_timedelta("14:30:00")
HORA_FIN_T2 = pd.to_timedelta("23:00:00")
MAX_EXTRA = pd.Timedelta(hours=3)
PASARELA_SET = {"A1", "A2", "A3", "A4", "B1", "B2", "B3", "B4", "C1", "C2", "C3", "C4"}


def clasificar_turno(ts):
    hora = ts - ts.normalize()
    if HORA_INICIO_T1 <= hora < HORA_FIN_T1:
        return "Turno 1"
    if HORA_FIN_T1 <= hora < HORA_FIN_T2:
        return "Turno 2"
    return "Turno 3"


def fecha_operativa(ts, turno):
    # El turno 3 puede iniciar antes de medianoche y terminar al dia siguiente.
    if turno == "Turno 3" and (ts - ts.normalize()) < HORA_INICIO_T1:
        return (ts - pd.Timedelta(days=1)).normalize()
    return ts.normalize()


def fin_turno(fecha_op, turno):
    if turno == "Turno 1":
        return fecha_op + HORA_FIN_T1
    if turno == "Turno 2":
        return fecha_op + HORA_FIN_T2
    return fecha_op + pd.Timedelta(days=1) + HORA_INICIO_T1


def ordenar_pasillos(serie_pasillos):
    vals = [str(v).strip() for v in serie_pasillos.dropna().unique() if str(v).strip() != ""]

    def llave(x):
        try:
            return (0, int(float(x)))
        except ValueError:
            return (1, x)

    vals = sorted(vals, key=llave)
    return ", ".join(vals)


def tokenizar_pasillos(pasillos_str):
    if pd.isna(pasillos_str) or str(pasillos_str).strip() == "":
        return set()
    return {p.strip().upper() for p in str(pasillos_str).split(",") if p.strip() != ""}


def combinar_pasillos(serie_pasillos):
    tokens = set()
    for valor in serie_pasillos.dropna():
        tokens |= tokenizar_pasillos(valor)

    def llave(x):
        # Orden natural: numericos primero, luego alfanumericos
        try:
            return (0, int(float(x)))
        except ValueError:
            return (1, x)

    return ", ".join(sorted(tokens, key=llave)) if tokens else ""


def clasificar_zona(pasillos_str):
    tokens = tokenizar_pasillos(pasillos_str)
    if not tokens:
        return pd.NA

    tiene_pasarela = len(tokens & PASARELA_SET) > 0
    solo_pasarela = tokens.issubset(PASARELA_SET)

    if solo_pasarela:
        return "Pasarela"
    if not tiene_pasarela:
        return "Selectivo"
    return "Combiando"


def piso_flag(pasillos_str, prefijo):
    tokens = tokenizar_pasillos(pasillos_str)
    return "Si" if any(t.startswith(prefijo) for t in tokens) else "No"


# Base limpia para segmentacion
base = df_surtido.copy()
base = base.dropna(subset=["fecha_hora", "usuario"])
base = base.sort_values(["usuario", "fecha_hora"]).reset_index(drop=True)

# Segmento continuo: nueva sesion cuando hay salto > 3h entre escaneos del mismo usuario
base["prev_fecha"] = base.groupby("usuario")["fecha_hora"].shift(1)
base["gap"] = base["fecha_hora"] - base["prev_fecha"]
base["nueva_sesion"] = base["prev_fecha"].isna() | (base["gap"] > MAX_EXTRA)
base["session_id"] = base.groupby("usuario")["nueva_sesion"].cumsum()

# Datos de sesion: se clasifica por el primer escaneo
sesiones = (
    base.groupby(["usuario", "session_id"], as_index=False)
    .agg(inicio=("fecha_hora", "min"), fin=("fecha_hora", "max"))
)

sesiones["turno"] = sesiones["inicio"].apply(clasificar_turno)
sesiones["fecha_operativa"] = sesiones.apply(lambda r: fecha_operativa(r["inicio"], r["turno"]), axis=1)
sesiones["fin_turno"] = sesiones.apply(lambda r: fin_turno(r["fecha_operativa"], r["turno"]), axis=1)

# Duracion real del bloque continuo
sesiones["duracion"] = sesiones["fin"] - sesiones["inicio"]

# Horas extra: ultimo escaneo despues del corte y hasta 3h despues
sesiones["delta_fin_turno"] = sesiones["fin"] - sesiones["fin_turno"]
sesiones["horas_extra"] = (sesiones["delta_fin_turno"] > pd.Timedelta(0)) & (sesiones["delta_fin_turno"] <= MAX_EXTRA)
sesiones["exceso_mayor_3h"] = sesiones["delta_fin_turno"] > MAX_EXTRA
sesiones["minutos_extra"] = np.where(sesiones["horas_extra"], sesiones["delta_fin_turno"].dt.total_seconds() / 60, 0).astype(int)

# Pasillos por sesion (string separado por comas)
pasillos_sesion = (
    base.groupby(["usuario", "session_id"])["pasillo"]
    .apply(ordenar_pasillos)
    .reset_index(name="pasillos")
)

# Total de codigos escaneados por sesion
codigos_sesion = (
    base.groupby(["usuario", "session_id"])["codigo"]
    .count()
    .reset_index(name="total_codigos")
)

# DF a nivel sesion
df_sesiones_usuario = sesiones.merge(pasillos_sesion, on=["usuario", "session_id"], how="left")
df_sesiones_usuario = df_sesiones_usuario.merge(codigos_sesion, on=["usuario", "session_id"], how="left")
df_sesiones_usuario["dia"] = df_sesiones_usuario["fecha_operativa"].dt.date

# Consolidar en un solo registro por usuario + dia + turno
# (evita duplicados cuando un usuario tiene varias sesiones dentro del mismo turno)
df_turnos_usuario = (
    df_sesiones_usuario.groupby(["dia", "usuario", "turno"], as_index=False)
    .agg(
        inicio=("inicio", "min"),
        fin=("fin", "max"),
        duracion=("duracion", "sum"),
        total_codigos=("total_codigos", "sum"),
        pasillos=("pasillos", combinar_pasillos),
        horas_extra=("horas_extra", "max"),
        minutos_extra=("minutos_extra", "sum"),
        exceso_mayor_3h=("exceso_mayor_3h", "max"),
    )
)

df_turnos_usuario["zona"] = df_turnos_usuario["pasillos"].apply(clasificar_zona)
df_turnos_usuario["piso_a"] = df_turnos_usuario["pasillos"].apply(lambda x: piso_flag(x, "A"))
df_turnos_usuario["piso_b"] = df_turnos_usuario["pasillos"].apply(lambda x: piso_flag(x, "B"))
df_turnos_usuario["piso_c"] = df_turnos_usuario["pasillos"].apply(lambda x: piso_flag(x, "C"))

df_turnos_usuario = df_turnos_usuario[
    [
        "dia",
        "usuario",
        "turno",
        "inicio",
        "fin",
        "duracion",
        "total_codigos",
        "pasillos",
        "zona",
        "piso_a",
        "piso_b",
        "piso_c",
        "horas_extra",
        "minutos_extra",
        "exceso_mayor_3h",
    ]
].sort_values(["dia", "turno", "usuario", "inicio"]).reset_index(drop=True)

print("DF creado: df_turnos_usuario (consolidado por dia+usuario+turno)")
print(df_turnos_usuario.head(20))
print("\nTotal registros:", len(df_turnos_usuario))

DF creado: df_turnos_usuario (consolidado por dia+usuario+turno)
           dia                    usuario    turno              inicio  \
0   2026-01-01     4003715 4003715 GARCIA  Turno 2 2026-01-01 16:28:39   
1   2026-01-01   4003903 4003903 MONTALVO  Turno 2 2026-01-01 16:33:42   
2   2026-01-01     4006242 4006242 COZATL  Turno 2 2026-01-01 16:42:54   
3   2026-01-01      4013155 4013155 ORTIZ  Turno 2 2026-01-01 16:29:04   
4   2026-01-01      4014237 4014237 REYES  Turno 2 2026-01-01 16:33:19   
5   2026-01-01   4022581 4022581 GONZALEZ  Turno 2 2026-01-01 16:25:35   
6   2026-01-01    4030588 4030588 VAZQUEZ  Turno 2 2026-01-01 16:27:10   
7   2026-01-01    4030751 4030751 CABRERA  Turno 2 2026-01-01 16:29:24   
8   2026-01-01  4031603 4031603 CHAVARRIA  Turno 2 2026-01-01 16:20:48   
9   2026-01-01     4032356 4032356 HUERTA  Turno 2 2026-01-01 16:31:36   
10  2026-01-01       4032668 4032668 DIAZ  Turno 2 2026-01-01 16:24:50   
11  2026-01-01     4032670 4032670 FLORES  Turn

In [3]:
# Exportar resultados a XLSX con nombre dinámico (timestamp para cada ejecución)
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ruta_salida = Path(r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 2. PROYECTOS\HANDHELDS SPUE\Pruebas\Abril_Honeywell_Urovo+T1Pro") / f"Analisis_turnos_surtido_{timestamp}.xlsx"


def fmt_duracion(td):
    if pd.isna(td):
        return ""
    total = int(td.total_seconds())
    h = total // 3600
    m = (total % 3600) // 60
    s = total % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


df_export = df_turnos_usuario.copy()

# Convertir a datetime timezone-naive
df_export["dia"] = pd.to_datetime(df_export["dia"])
df_export["inicio"] = pd.to_datetime(df_export["inicio"]).dt.tz_localize(None)
df_export["fin"] = pd.to_datetime(df_export["fin"]).dt.tz_localize(None)

# Duracion como hh:mm:ss
df_export["duracion"] = df_export["duracion"].apply(fmt_duracion)

with pd.ExcelWriter(ruta_salida, engine="openpyxl") as writer:
    df_export.to_excel(writer, sheet_name="Turnos_usuario", index=False)
    df_resumen_columnas.to_excel(writer, sheet_name="Resumen_columnas", index=False)

    ws = writer.sheets["Turnos_usuario"]
    col_dia = df_export.columns.get_loc("dia") + 1
    col_inicio = df_export.columns.get_loc("inicio") + 1
    col_fin = df_export.columns.get_loc("fin") + 1

    # Escribir como datetime de Python para que openpyxl use serial number (no texto)
    for idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row)):
        dia_val = df_export["dia"].iloc[idx]
        if pd.notna(dia_val):
            row[col_dia - 1].value = dia_val.to_pydatetime()
        row[col_dia - 1].number_format = "DD/MM/YYYY"

        inicio_val = df_export["inicio"].iloc[idx]
        if pd.notna(inicio_val):
            row[col_inicio - 1].value = inicio_val.to_pydatetime()
        row[col_inicio - 1].number_format = "DD/MM/YYYY HH:MM:SS"

        fin_val = df_export["fin"].iloc[idx]
        if pd.notna(fin_val):
            row[col_fin - 1].value = fin_val.to_pydatetime()
        row[col_fin - 1].number_format = "DD/MM/YYYY HH:MM:SS"

print("Archivo generado:", ruta_salida)
print("Filas exportadas (Turnos_usuario):", len(df_export))

Archivo generado: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 2. PROYECTOS\HANDHELDS SPUE\Pruebas\Abril_Honeywell_Urovo+T1Pro\Analisis_turnos_surtido_20260422_074246.xlsx
Filas exportadas (Turnos_usuario): 10738


In [4]:
# Diagnóstico rápido de dia antes de exportar
_d = pd.to_datetime(df_turnos_usuario["dia"], errors="coerce")
print("min dia:", _d.min())
print("max dia:", _d.max())
print("meses:", sorted(_d.dt.month.dropna().unique().tolist()))
print("primeros 15 dias:", _d.dt.strftime("%d/%m/%Y").dropna().head(15).tolist())

min dia: 2026-01-01 00:00:00
max dia: 2026-04-20 00:00:00
meses: [1, 2, 3, 4]
primeros 15 dias: ['01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026', '01/01/2026']
